# Chapter 13: Aliens!

> En este capítulo se agregan los aliens a Alien Invasion: primero uno, luego una flota completa que se mueve, se destruye con las balas y termina el juego cuando el jugador se queda sin naves.

## Summary

Se crea la clase `Alien` (parecida a `Ship`) y se genera una **flota** de aliens con dos `while` anidados: el interno ubica los aliens en una fila (valores de x) y el externo agrega filas (valores de y).
La flota se mueve con un método `update()` en cada alien; cuando uno toca un borde, toda la flota baja y cambia de dirección (`fleet_direction` vale `1` o `-1`).
Las colisiones se detectan con `pygame.sprite.groupcollide()` (balas contra aliens) y `pygame.sprite.spritecollideany()` (aliens contra la nave).
Cuando se destruye la flota, aparece una nueva; cuando un alien toca la nave o llega al borde inferior, se pierde una nave, que se lleva en una clase `GameStats`.
Un flag `game_active` termina el juego cuando se acaban las naves.
Se sigue haciendo refactoring: `_create_alien()`, `_check_fleet_edges()`, `_change_fleet_direction()`, `_check_bullet_alien_collisions()`, `_ship_hit()` y `_check_aliens_bottom()`.

## Cheat Sheet

### Funciones, métodos y atributos

| Quiero... | Sintaxis | Ejemplo | Resultado / nota |
|---|---|---|---|
| Crear la clase de un alien | `class Alien(Sprite):` | `from pygame.sprite import Sprite` | igual que `Ship`, pero hereda de `Sprite` para poder agrupar los aliens |
| Ubicar el primer alien | `self.rect.x = self.rect.width` | `self.rect.y = self.rect.height` | deja un espacio igual a su ancho y su alto |
| Guardar un grupo de aliens | `pygame.sprite.Group()` | `self.aliens = pygame.sprite.Group()` | |
| Dibujar todo un grupo | `grupo.draw(superficie)` | `self.aliens.draw(self.screen)` | dibuja cada sprite en la posición de su `rect`; no hace falta un método de dibujo en `Alien` |
| Ancho y alto de un rect | `rect.size` | `alien_width, alien_height = alien.rect.size` | tupla `(ancho, alto)` |
| Actualizar todo un grupo | `grupo.update()` | `self.aliens.update()` | llama a `update()` de cada alien |
| Saber si un alien está en un borde | `check_edges()` | `return (self.rect.right >= screen_rect.right) or (self.rect.left <= 0)` | devuelve `True` / `False` |
| Cambiar de dirección | `self.settings.fleet_direction *= -1` | | alterna entre `1` (derecha) y `-1` (izquierda) |
| Detectar balas que chocan con aliens | `pygame.sprite.groupcollide(g1, g2, True, True)` | `pygame.sprite.groupcollide(self.bullets, self.aliens, True, True)` | devuelve un diccionario: llave = bala, valor = lista de aliens golpeados; los `True` borran ambos |
| Detectar un alien que choca con la nave | `pygame.sprite.spritecollideany(sprite, grupo)` | `pygame.sprite.spritecollideany(self.ship, self.aliens)` | devuelve el primer alien que choca, o `None` |
| Saber si un grupo está vacío | `if not grupo:` | `if not self.aliens:` | un grupo vacío es `False` |
| Vaciar un grupo | `grupo.empty()` | `self.bullets.empty()` | quita todos los sprites |
| Pausar el programa | `sleep(segundos)` | `sleep(0.5)` | `from time import sleep` |
| Detectar el borde inferior | `alien.rect.bottom >= screen_height` | `self.settings.screen_height` | |
| Número aleatorio (ejercicio 13-2) | `randint(a, b)` | `from random import randint` | entero entre `a` y `b` |

### Estructuras de programa

| Quiero... | Patrón | Nota |
|---|---|---|
| Crear una fila de aliens | `while current_x < (screen_width - 2 * alien_width):` | agregar un alien y sumar `2 * alien_width` a `current_x` |
| Crear varias filas | un `while` sobre `current_y` que contiene al `while` de `current_x` | al terminar cada fila, `current_x` vuelve al inicio y `current_y` aumenta |
| Llevar estadísticas | clase `GameStats` con `reset_stats()` | se llama a `reset_stats()` desde `__init__()` y cada vez que empieza una partida |
| Terminar el juego | flag `self.game_active` | `_check_events()` y `_update_screen()` corren siempre; las actualizaciones solo si el juego está activo |

### Settings nuevos

| Setting | Valor | Para qué sirve |
|---|---|---|
| `alien_speed` | `1.0` | velocidad horizontal de cada alien |
| `fleet_drop_speed` | `10` | cuánto baja la flota al tocar un borde |
| `fleet_direction` | `1` | `1` = derecha, `-1` = izquierda |
| `ship_limit` | `3` | cantidad de naves del jugador |
| `bullet_speed` | `2.5` | un poco más rápida que antes |

### Problemas del capítulo

| Problema | Mensaje | Causa | Solución |
|---|---|---|---|
| La flota se mueve pero no baja ni cambia de dirección | (no hay mensaje) | falta llamar a `_check_fleet_edges()` antes de `self.aliens.update()` | llamarlo al inicio de `_update_aliens()` |
| La flota no se ve bien | (no hay mensaje) | mala indentación en los `while` anidados | las dos últimas líneas van dentro del `while` externo y fuera del interno |
| Al destruir un alien se destruyen más de uno por error | (no hay mensaje) | `groupcollide()` con balas muy anchas | restaurar `bullet_width` después de probar |
| `ships_left` queda negativo | (no hay mensaje) | no hay un flag que termine el juego | usar `game_active` |

## Reviewing the Project

Al empezar una nueva fase de un proyecto grande, conviene revisar el plan. En este capítulo se va a:

- Agregar un alien arriba a la izquierda, con espacio a su alrededor.
- Llenar la parte superior de la pantalla con tantos aliens como quepan horizontalmente y luego crear más filas hasta tener la flota completa.
- Hacer que la flota se mueva hacia los lados y hacia abajo hasta que se destruya completa, un alien toque la nave o un alien llegue al suelo. Si se destruye la flota, se crea una nueva; si un alien toca la nave o el suelo, se destruye la nave y se crea una flota nueva.
- Limitar el número de naves y terminar el juego cuando se acaben.

También se revisa el código existente al empezar una fase nueva. Como se ha hecho refactoring en el camino, aquí no hace falta.

## Creating the First Alien

Poner un alien en la pantalla es parecido a poner la nave. Su comportamiento lo controla una clase `Alien`, estructurada como `Ship`. Se siguen usando imágenes `.bmp` (`alien.bmp`, en los recursos del libro), con un fondo gris que coincide con el de la pantalla, y se guarda en la carpeta `images`.

### Creating the Alien Class

La clase se guarda en `alien.py`:

In [ ]:
# alien.py
import pygame
from pygame.sprite import Sprite

class Alien(Sprite):
    """A class to represent a single alien in the fleet."""

    def __init__(self, ai_game):
        """Initialize the alien and set its starting position."""
        super().__init__()
        self.screen = ai_game.screen

        # Load the alien image and set its rect attribute.
        self.image = pygame.image.load('images/alien.bmp')
        self.rect = self.image.get_rect()

        # Start each new alien near the top left of the screen.
        self.rect.x = self.rect.width
        self.rect.y = self.rect.height

        # Store the alien's exact horizontal position.
        self.x = float(self.rect.x)

Casi todo es igual que en `Ship`, salvo la posición inicial: cada alien se ubica cerca de la esquina superior izquierda, con un espacio a la izquierda igual a su ancho y uno arriba igual a su alto, para que se vea bien.
Lo que más importa es la velocidad horizontal, así que se guarda la posición horizontal exacta en `self.x`.
`Alien` no necesita un método para dibujarse: se usa un método de los grupos de Pygame que dibuja todos sus elementos automáticamente.

### Creating an Instance of the Alien

Se crea un alien al final de `__init__()` de `AlienInvasion`, en un nuevo helper method, `_create_fleet()`, porque después serán muchos. El orden de los métodos en una clase no importa, mientras haya consistencia: se coloca justo antes de `_update_screen()`.

In [ ]:
# alien_invasion.py
# --snip--
from bullet import Bullet
from alien import Alien

In [ ]:
# alien_invasion.py
    def __init__(self):
        # --snip--
        self.ship = Ship(self)
        self.bullets = pygame.sprite.Group()
        self.aliens = pygame.sprite.Group()

        self._create_fleet()

In [ ]:
# alien_invasion.py
    def _create_fleet(self):
        """Create the fleet of aliens."""
        # Make an alien.
        alien = Alien(self)
        self.aliens.add(alien)

Para que el alien aparezca, se llama al método `draw()` del grupo en `_update_screen()`:

In [ ]:
# alien_invasion.py
    def _update_screen(self):
        # --snip--
        self.ship.blitme()
        self.aliens.draw(self.screen)

        pygame.display.flip()

`draw()` dibuja cada elemento del grupo en la posición definida por su `rect`. Requiere un argumento: la superficie donde dibujar.

### Comparison: Creating the First Alien

| | `Ship` | `Alien` |
|---|---|---|
| Hereda de | nada (hasta el Capítulo 14) | `Sprite` |
| Imagen | `images/ship.bmp` | `images/alien.bmp` |
| Posición inicial | abajo al centro (`midbottom`) | arriba a la izquierda (`rect.x = rect.width`, `rect.y = rect.height`) |
| Cómo se dibuja | `blitme()` | `self.aliens.draw(self.screen)` (método del grupo) |
| Cantidad | una instancia | muchas, en un `Group` |

## Building the Alien Fleet

Hay que llenar la parte superior de la pantalla sin saturarla. Se hace agregando aliens a lo largo de la pantalla hasta que no quede espacio, y repitiendo con más filas mientras haya espacio vertical.

### Creating a Row of Aliens

Primero se crea un alien para conocer su ancho. Luego se ubica a la izquierda y se siguen agregando aliens mientras haya espacio:

In [ ]:
# alien_invasion.py
    def _create_fleet(self):
        """Create the fleet of aliens."""
        # Create an alien and keep adding aliens until there's no room left.
        # Spacing between aliens is one alien width.
        alien = Alien(self)
        alien_width = alien.rect.width

        current_x = alien_width
        while current_x < (self.settings.screen_width - 2 * alien_width):
            new_alien = Alien(self)
            new_alien.x = current_x
            new_alien.rect.x = current_x
            self.aliens.add(new_alien)
            current_x += 2 * alien_width

- `current_x` es la posición horizontal del próximo alien. Empieza en un ancho de alien, para separar el primero del borde izquierdo.
- El `while` sigue mientras haya espacio. Si la condición fuera `current_x < screen_width`, el último alien quedaría pegado al borde derecho; por eso se deja un margen de **dos anchos de alien** a la derecha.
- En cada vuelta se crea un alien, se fija su posición exacta (`new_alien.x`) y la de su rect, y se agrega al grupo.
- Se suman dos anchos de alien a `current_x`: uno para pasar el alien recién puesto y otro para dejar espacio entre ellos.

> No siempre es obvio cómo armar un loop así. Las ideas iniciales no tienen que ser correctas: se puede empezar con los aliens mal ubicados y modificar el loop hasta lograr el espacio adecuado.

### Refactoring _create_fleet()

Como todavía hay más trabajo, se limpia el método con un helper, `_create_alien()`, que se llama desde `_create_fleet()`:

In [ ]:
# alien_invasion.py
    def _create_fleet(self):
        # --snip--
        while current_x < (self.settings.screen_width - 2 * alien_width):
            self._create_alien(current_x)
            current_x += 2 * alien_width

    def _create_alien(self, x_position):
        """Create an alien and place it in the row."""
        new_alien = Alien(self)
        new_alien.x = x_position
        new_alien.rect.x = x_position
        self.aliens.add(new_alien)

`_create_alien()` recibe además de `self` el valor x donde se debe ubicar el alien. Su cuerpo es el mismo código que estaba en `_create_fleet()`, usando `x_position` en vez de `current_x`.

### Adding Rows

Para terminar la flota se agregan filas hasta que se acabe el espacio, con un **loop anidado**: un `while` externo ubica filas según los valores de y, y el interno ubica los aliens de cada fila según los valores de x. Se paran las filas cerca del borde inferior, dejando espacio para la nave y para disparar.

In [ ]:
# alien_invasion.py
    def _create_fleet(self):
        """Create the fleet of aliens."""
        # Create an alien and keep adding aliens until there's no room left.
        # Spacing between aliens is one alien width and one alien height.
        alien = Alien(self)
        alien_width, alien_height = alien.rect.size

        current_x, current_y = alien_width, alien_height
        while current_y < (self.settings.screen_height - 3 * alien_height):
            while current_x < (self.settings.screen_width - 2 * alien_width):
                self._create_alien(current_x, current_y)
                current_x += 2 * alien_width

            # Finished a row; reset x value, and increment y value.
            current_x = alien_width
            current_y += 2 * alien_height

- `alien.rect.size` es una tupla con el ancho y el alto del rect.
- `current_x, current_y` son los valores iniciales de la primera posición: un ancho desde la izquierda y un alto desde arriba.
- El `while` externo sigue mientras el y de la siguiente fila sea menor que el alto de la pantalla menos **tres altos de alien**. Si el espacio no queda bien, se puede ajustar.
- **La indentación importa**: las dos últimas líneas están dentro del `while` externo pero fuera del interno, así que se ejecutan una vez por fila. Se reinicia `current_x` y se suman dos altos de alien a `current_y`.

`_create_alien()` ahora recibe también el valor de y:

In [ ]:
# alien_invasion.py
    def _create_alien(self, x_position, y_position):
        """Create an alien and place it in the fleet."""
        new_alien = Alien(self)
        new_alien.x = x_position
        new_alien.rect.x = x_position
        new_alien.rect.y = y_position
        self.aliens.add(new_alien)

Al ejecutar el juego se ve la flota completa.

### Comparison: Building the Alien Fleet

| | `while` interno | `while` externo |
|---|---|---|
| Controla | las **columnas** de una fila | las **filas** |
| Variable | `current_x` | `current_y` |
| Condición | `current_x < (screen_width - 2 * alien_width)` | `current_y < (screen_height - 3 * alien_height)` |
| Avanza | `current_x += 2 * alien_width` | `current_y += 2 * alien_height` |
| Cuándo se reinicia `current_x` | — | al terminar cada fila (`current_x = alien_width`) |

| | `_create_fleet()` | `_create_alien()` |
|---|---|---|
| Qué hace | decide **dónde** va cada alien (loops anidados) | crea **un** alien y lo agrega al grupo |
| Recibe | solo `self` | `x_position` y `y_position` |

### Try it Yourself

> 13-1. Stars:
Find an image of a star. Make a grid of stars appear on the screen.

> 13-2. Better Stars:
You can make a more realistic star pattern by introducing randomness when you place each star. Recall from Chapter 9 that you can get a random number like this:

```python
from random import randint
random_number = randint(-10, 10)
```

This code returns a random integer between −10 and 10. Using your code in Exercise 13-1, adjust each star’s position by a random amount.

In [ ]:
# 13-1


In [ ]:
# 13-2


## Making the Fleet Move

Ahora la flota se mueve hacia la derecha hasta tocar el borde, baja una cantidad fija y cambia de dirección. Esto se repite hasta que todos los aliens se destruyan, uno choque con la nave o uno llegue al borde inferior.

### Moving the Aliens Right

Para moverlos se usa un método `update()` en `alien.py`, que se llama para cada alien del grupo. Primero, un setting para la velocidad:

In [ ]:
# settings.py
    def __init__(self):
        # --snip--
        # Alien settings
        self.alien_speed = 1.0

In [ ]:
# alien.py
    def __init__(self, ai_game):
        """Initialize the alien and set its starting position."""
        super().__init__()
        self.screen = ai_game.screen
        self.settings = ai_game.settings
        # --snip--
    def update(self):
        """Move the alien to the right."""
        self.x += self.settings.alien_speed
        self.rect.x = self.x

Se crea un atributo `settings` para acceder a la velocidad. Se mueve la posición exacta `self.x` (que admite decimales) y luego se usa para actualizar el rect. En el loop principal se agrega la llamada a un método nuevo, `_update_aliens()`, **después** de actualizar las balas, porque pronto se revisarán las colisiones entre balas y aliens:

In [ ]:
# alien_invasion.py
        while True:
            self._check_events()
            self.ship.update()
            self._update_bullets()
            self._update_aliens()
            self._update_screen()
            self.clock.tick(60)

In [ ]:
# alien_invasion.py
    def _update_aliens(self):
        """Update the positions of all aliens in the fleet."""
        self.aliens.update()

`self.aliens.update()` llama al `update()` de cada alien. Al ejecutarlo, la flota se mueve a la derecha y desaparece por el lado de la pantalla.

### Creating Settings for Fleet Direction

Se agregan los settings para que la flota baje y vaya hacia la izquierda al tocar el borde derecho:

In [ ]:
# settings.py
        # Alien settings
        self.alien_speed = 1.0
        self.fleet_drop_speed = 10
        # fleet_direction of 1 represents right; -1 represents left.
        self.fleet_direction = 1

- `fleet_drop_speed` controla cuánto baja la flota cada vez que un alien toca un borde. Se separa de la velocidad horizontal para ajustarlas por separado.
- Para `fleet_direction` se podría usar un texto como `'left'` o `'right'`, pero habría que usar `if-elif` para revisar la dirección. Como solo hay dos direcciones, se usan los valores **`1` y `-1`** y se alternan. Además tiene sentido porque ir a la derecha es *sumar* a la coordenada x y a la izquierda es *restar*.

### Checking Whether an Alien Has Hit the Edge

Se necesita un método para saber si un alien está en un borde, y `update()` debe mover cada alien en la dirección correcta. Ambos son parte de la clase `Alien`:

In [ ]:
# alien.py
    def check_edges(self):
        """Return True if alien is at edge of screen."""
        screen_rect = self.screen.get_rect()
        return (self.rect.right >= screen_rect.right) or (self.rect.left <= 0)

    def update(self):
        """Move the alien right or left."""
        self.x += self.settings.alien_speed * self.settings.fleet_direction
        self.rect.x = self.x

- El alien está en el borde derecho si el `right` de su rect es mayor o igual que el `right` del rect de la pantalla, y en el borde izquierdo si su `left` es menor o igual que `0`.
- En vez de poner la condición dentro de un `if`, se pone directamente en el `return`: el método devuelve `True` si el alien está en cualquiera de los bordes, y `False` si no.
- En `update()`, la velocidad se multiplica por `fleet_direction`: con `1` se suma (derecha) y con `-1` se resta (izquierda).

### Dropping the Fleet and Changing Direction

Cuando un alien llega al borde, toda la flota baja y cambia de dirección. Esto se revisa en `AlienInvasion`, con dos métodos nuevos: `_check_fleet_edges()` y `_change_fleet_direction()`.

In [ ]:
# alien_invasion.py
    def _check_fleet_edges(self):
        """Respond appropriately if any aliens have reached an edge."""
        for alien in self.aliens.sprites():
            if alien.check_edges():
                self._change_fleet_direction()
                break

    def _change_fleet_direction(self):
        """Drop the entire fleet and change the fleet's direction."""
        for alien in self.aliens.sprites():
            alien.rect.y += self.settings.fleet_drop_speed
        self.settings.fleet_direction *= -1

- `_check_fleet_edges()` recorre la flota y llama a `check_edges()` en cada alien. Si alguno devuelve `True`, llama a `_change_fleet_direction()` y sale del loop con `break`.
- `_change_fleet_direction()` baja a cada alien con `fleet_drop_speed` y cambia la dirección **una sola vez**, multiplicando `fleet_direction` por `-1`. Esa línea no va dentro del `for`: se quiere cambiar la posición vertical de cada alien, pero la dirección de la flota solo una vez.

In [ ]:
# alien_invasion.py
    def _update_aliens(self):
        """Check if the fleet is at an edge, then update positions."""
        self._check_fleet_edges()
        self.aliens.update()

Se llama a `_check_fleet_edges()` antes de actualizar la posición de cada alien. Ahora la flota rebota entre los bordes y baja cada vez que toca uno.

### Comparison: Making the Fleet Move

| | Texto (`'left'` / `'right'`) | Número (`1` / `-1`) |
|---|---|---|
| Para saber hacia dónde mover | hay que usar `if-elif` | se multiplica la velocidad por la dirección |
| Cambiar de dirección | asignar el otro texto | `fleet_direction *= -1` |
| En el libro | no se usa | **se usa** |

| | `fleet_drop_speed` | `alien_speed` |
|---|---|---|
| Eje | vertical (baja) | horizontal (izquierda / derecha) |
| Se aplica | una vez, al tocar un borde | en cada vuelta del loop |
| Valor inicial | `10` | `1.0` |

| | `Alien.check_edges()` | `AlienInvasion._check_fleet_edges()` |
|---|---|---|
| Revisa | **un** alien | **toda** la flota |
| Devuelve / hace | devuelve `True` / `False` | llama a `_change_fleet_direction()` si algún alien está en un borde |

### Try it Yourself

> 13-3. Raindrops:
Find an image of a raindrop and create a grid of raindrops. Make the raindrops fall toward the bottom of the screen until they disappear.

> 13-4. Steady Rain:
Modify your code in Exercise 13-3 so when a row of raindrops disappears off the bottom of the screen, a new row appears at the top of the screen and begins to fall.

In [ ]:
# 13-3


In [ ]:
# 13-4


## Shooting Aliens

Por ahora las balas atraviesan a los aliens porque no se revisan las colisiones. En programación de juegos, una **colisión** ocurre cuando dos elementos se superponen. Para que las balas destruyan aliens se usa `sprite.groupcollide()`, que busca colisiones entre los miembros de dos grupos.

### Detecting Bullet Collisions

Las colisiones se revisan justo después de actualizar la posición de las balas. `groupcollide()` compara el rect de cada bala con el de cada alien y devuelve un **diccionario** con las balas y aliens que chocaron: cada llave es una bala y el valor es el alien golpeado (este diccionario se usará para el puntaje en el Capítulo 14).

In [ ]:
# alien_invasion.py
    def _update_bullets(self):
        """Update position of bullets and get rid of old bullets."""
        # --snip--
        # Check for any bullets that have hit aliens.
        #   If so, get rid of the bullet and the alien.
        collisions = pygame.sprite.groupcollide(
                self.bullets, self.aliens, True, True)

Cuando los rects de una bala y un alien se superponen, `groupcollide()` agrega un par llave-valor al diccionario. Los dos `True` le dicen a Pygame que borre las balas y los aliens que chocaron.
(Para una bala muy potente, que atraviese y destruya todos los aliens, se puede poner el primer argumento booleano en `False` y dejar el segundo en `True`: los aliens golpeados desaparecen y las balas siguen hasta salir por arriba.)

### Making Larger Bullets for Testing

Algunas funciones son tediosas de probar en el juego normal. Se pueden cambiar settings para facilitar las pruebas: reducir la pantalla para que haya menos aliens, aumentar la velocidad de las balas o dar muchas balas a la vez.
Un cambio muy útil es usar **balas muy anchas** que siguen activas después de golpear un alien: por ejemplo `bullet_width = 300` o incluso `3000`. Al terminar de probar hay que restaurar los valores normales.

### Repopulating the Fleet

Los aliens son incansables: cada vez que se destruye la flota, debe aparecer una nueva. Se revisa si el grupo `aliens` está vacío al final de `_update_bullets()`, que es donde se destruyen los aliens:

In [ ]:
# alien_invasion.py
    def _update_bullets(self):
        # --snip--
        if not self.aliens:
            # Destroy existing bullets and create new fleet.
            self.bullets.empty()
            self._create_fleet()

Un grupo vacío se evalúa como `False`, así que `if not self.aliens:` es una forma simple de revisar si quedan aliens. Si no quedan, `empty()` quita todas las balas y `_create_fleet()` llena otra vez la pantalla.

### Speeding Up the Bullets

Si las balas se sienten muy lentas o muy rápidas, se ajusta `bullet_speed` en `settings.py` (en el libro se cambia a `2.5`). El mejor valor depende de la experiencia de juego, pero hay que recordar que el juego se va a ir acelerando, así que no debe ser demasiado rápido al principio.

In [ ]:
# settings.py
        # Bullet settings
        self.bullet_speed = 2.5
        self.bullet_width = 3
        # --snip--

### Refactoring _update_bullets()

Para que `_update_bullets()` no haga tantas cosas, el código de las colisiones se mueve a otro método:

In [ ]:
# alien_invasion.py
    def _update_bullets(self):
        # --snip--
        # Get rid of bullets that have disappeared.
        for bullet in self.bullets.copy():
            if bullet.rect.bottom <= 0:
                self.bullets.remove(bullet)

        self._check_bullet_alien_collisions()

    def _check_bullet_alien_collisions(self):
        """Respond to bullet-alien collisions."""
        # Remove any bullets and aliens that have collided.
        collisions = pygame.sprite.groupcollide(
                self.bullets, self.aliens, True, True)

        if not self.aliens:
            # Destroy existing bullets and create new fleet.
            self.bullets.empty()
            self._create_fleet()

`_check_bullet_alien_collisions()` busca colisiones y responde si se destruyó toda la flota. Así `_update_bullets()` no crece demasiado.

### Comparison: Shooting Aliens

| | `groupcollide(bullets, aliens, True, True)` | `groupcollide(bullets, aliens, False, True)` |
|---|---|---|
| Bala que choca | se borra | **sigue** viva |
| Alien que choca | se borra | se borra |
| Efecto | una bala destruye un alien | una bala potente destruye todos los aliens que atraviesa |

| | `groupcollide()` | `spritecollideany()` |
|---|---|---|
| Compara | dos **grupos** | un **sprite** contra un grupo |
| Devuelve | un diccionario (balas → aliens golpeados) | el primer alien que chocó, o `None` |
| En el juego | balas contra aliens | nave contra aliens (más adelante) |

| | `_update_bullets()` | `_check_bullet_alien_collisions()` |
|---|---|---|
| Qué hace | actualiza y borra las balas que salen por arriba | responde a las colisiones y crea una flota nueva |
| Llama al otro | sí, al final | no |

### Try it Yourself

> 13-5. Sideways Shooter Part 2:
We’ve come a long way since Exercise 12-6, Sideways Shooter. For this exercise, try to develop Sideways Shooter to the same point we’ve brought Alien Invasion to. Add a fleet of aliens, and make them move sideways toward the ship. Or, write code that places aliens at random positions along the right side of the screen and then sends them toward the ship. Also, write code that makes the aliens disappear when they’re hit.

In [ ]:
# 13-5


## Ending the Game

¿Qué gracia tiene un juego que no se puede perder? Si el jugador no destruye la flota a tiempo, los aliens destruirán la nave al tocarla. Además se limita el número de naves y se destruye la nave si un alien llega al borde inferior. El juego termina cuando se acaban las naves.

### Detecting Alien-Ship Collisions

Primero se revisan las colisiones entre aliens y la nave, justo después de actualizar la posición de los aliens:

In [ ]:
# alien_invasion.py
    def _update_aliens(self):
        # --snip--
        self.aliens.update()

        # Look for alien-ship collisions.
        if pygame.sprite.spritecollideany(self.ship, self.aliens):
            print("Ship hit!!!")

`spritecollideany()` recibe un sprite y un grupo, y busca cualquier miembro del grupo que haya chocado con el sprite. Se detiene en cuanto encuentra uno y lo devuelve; si no hay colisiones, devuelve `None` y el `if` no se ejecuta.
El `print()` sirve para comprobar que la detección funciona antes de escribir el resto del código. Para probarlo más rápido conviene subir `fleet_drop_speed` a 50 o 100, para que los aliens lleguen antes a la nave.

### Responding to Alien-Ship Collisions

En vez de destruir la nave y crear otra, se **cuentan las veces que ha sido golpeada** con estadísticas del juego (que también servirán para el puntaje). Se escribe una clase `GameStats`, guardada en `game_stats.py`:

In [ ]:
# game_stats.py
class GameStats:
    """Track statistics for Alien Invasion."""

    def __init__(self, ai_game):
        """Initialize statistics."""
        self.settings = ai_game.settings
        self.reset_stats()

    def reset_stats(self):
        """Initialize statistics that can change during the game."""
        self.ships_left = self.settings.ship_limit

Se crea una sola instancia de `GameStats` mientras el juego corre, pero algunas estadísticas hay que reiniciarlas cada vez que el jugador empieza una partida nueva. Por eso se inicializan en `reset_stats()` en vez de `__init__()`, y se llama a `reset_stats()` desde `__init__()` para que queden bien al crear la instancia.
Por ahora hay una sola estadística: `ships_left`. El número inicial de naves se guarda en `settings.py` como `ship_limit`:

In [ ]:
# settings.py
        # Ship settings
        self.ship_speed = 1.5
        self.ship_limit = 3

En `alien_invasion.py` se actualizan los `import` (`sleep` para pausar el juego cuando la nave es golpeada, y `GameStats`) y se crea la instancia después de crear la ventana pero antes de los demás elementos:

In [ ]:
# alien_invasion.py
import sys
from time import sleep

import pygame

from settings import Settings
from game_stats import GameStats
from ship import Ship
# --snip--

In [ ]:
# alien_invasion.py
    def __init__(self):
        # --snip--
        pygame.display.set_caption("Alien Invasion")

        # Create an instance to store game statistics.
        self.stats = GameStats(self)

        self.ship = Ship(self)
        # --snip--

Cuando un alien golpea la nave se resta 1 a las naves restantes, se destruyen los aliens y las balas, se crea una flota nueva, se vuelve a centrar la nave y se pausa el juego un momento. Casi todo eso va en un método nuevo, `_ship_hit()`:

In [ ]:
# alien_invasion.py
    def _ship_hit(self):
        """Respond to the ship being hit by an alien."""
        # Decrement ships_left.
        self.stats.ships_left -= 1

        # Get rid of any remaining bullets and aliens.
        self.bullets.empty()
        self.aliens.empty()

        # Create a new fleet and center the ship.
        self._create_fleet()
        self.ship.center_ship()

        # Pause.
        sleep(0.5)

- Se resta 1 a `ships_left` y se vacían los grupos `bullets` y `aliens`.
- Se crea una flota nueva y se centra la nave (con `center_ship()`, que se agrega a `Ship` enseguida).
- `sleep(0.5)` pausa el programa medio segundo, lo suficiente para que el jugador vea que la nave fue golpeada. Luego se pasa a `_update_screen()`, que dibuja la flota nueva.

In [ ]:
# alien_invasion.py
    def _update_aliens(self):
        # --snip--
        if pygame.sprite.spritecollideany(self.ship, self.aliens):
            self._ship_hit()

In [ ]:
# ship.py
    def center_ship(self):
        """Center the ship on the screen."""
        self.rect.midbottom = self.screen_rect.midbottom
        self.x = float(self.rect.x)

`center_ship()` centra la nave igual que en `__init__()` y además reinicia `self.x` para seguir su posición exacta.

> Nunca se hace más de una nave: se crea **una sola instancia** de `Ship` para todo el juego y se vuelve a centrar cada vez que es golpeada. La estadística `ships_left` indica cuándo el jugador se quedó sin naves.

### Aliens That Reach the Bottom of the Screen

Si un alien llega al borde inferior, el juego responde igual que cuando golpea la nave. Se agrega un método:

In [ ]:
# alien_invasion.py
    def _check_aliens_bottom(self):
        """Check if any aliens have reached the bottom of the screen."""
        for alien in self.aliens.sprites():
            if alien.rect.bottom >= self.settings.screen_height:
                # Treat this the same as if the ship got hit.
                self._ship_hit()
                break

Un alien llegó abajo si el `bottom` de su rect es mayor o igual que el alto de la pantalla. Con un solo alien basta: se llama a `_ship_hit()` y se sale del loop con `break`. Se llama a este método desde `_update_aliens()`, después de actualizar la posición de todos los aliens y de revisar las colisiones con la nave:

In [ ]:
# alien_invasion.py
    def _update_aliens(self):
        # --snip--
        # Look for alien-ship collisions.
        if pygame.sprite.spritecollideany(self.ship, self.aliens):
            self._ship_hit()

        # Look for aliens hitting the bottom of the screen.
        self._check_aliens_bottom()

### Game Over!

El juego nunca termina: `ships_left` se vuelve cada vez más negativo. Se agrega un flag `game_active` al final de `__init__()` en `AlienInvasion`:

In [ ]:
# alien_invasion.py
    def __init__(self):
        # --snip--
        # Start Alien Invasion in an active state.
        self.game_active = True

Y en `_ship_hit()` se pone en `False` cuando el jugador usó todas sus naves. Todo el código anterior pasa a un `if` que verifica que quede al menos una nave:

In [ ]:
# alien_invasion.py
    def _ship_hit(self):
        """Respond to ship being hit by alien."""
        if self.stats.ships_left > 0:
            # Decrement ships_left.
            self.stats.ships_left -= 1
            # --snip--
            # Pause.
            sleep(0.5)
        else:
            self.game_active = False

### Identifying When Parts of the Game Should Run

Hay que distinguir las partes del juego que corren siempre de las que corren solo cuando el juego está activo:

In [ ]:
# alien_invasion.py
    def run_game(self):
        """Start the main loop for the game."""
        while True:
            self._check_events()

            if self.game_active:
                self.ship.update()
                self._update_bullets()
                self._update_aliens()

            self._update_screen()
            self.clock.tick(60)

`_check_events()` se llama siempre, aunque el juego esté inactivo, porque hay que saber si el usuario presiona `Q` o cierra la ventana. También se sigue actualizando la pantalla, para poder cambiarla mientras se espera que el jugador decida empezar de nuevo. El resto solo se ejecuta cuando el juego está activo, porque si está inactivo no hace falta mover los elementos.
Al jugar, el juego se congela cuando se usaron todas las naves.

### Comparison: Ending the Game

| | `_ship_hit()` | `_check_aliens_bottom()` |
|---|---|---|
| Qué detecta | (no detecta: responde) | un alien que llegó al borde inferior |
| Qué hace | resta una nave, vacía grupos, crea una flota nueva, centra la nave, pausa | llama a `_ship_hit()` |
| Dónde se llama | desde `_update_aliens()` cuando un alien toca la nave, y desde `_check_aliens_bottom()` | desde `_update_aliens()` |

| | `GameStats.__init__()` | `GameStats.reset_stats()` |
|---|---|---|
| Cuándo se ejecuta | una vez, al crear el juego | al crear el juego **y** cada vez que empieza una partida |
| Qué guarda | (llama a `reset_stats()`) | datos que cambian en la partida (`ships_left`) |

| | `game_active = True` | `game_active = False` |
|---|---|---|
| `_check_events()` | corre | corre |
| `ship.update()`, `_update_bullets()`, `_update_aliens()` | corren | **no** corren |
| `_update_screen()` | corre | corre |

| | `Ship` (instancia) | `ships_left` (estadística) |
|---|---|---|
| Cuántas hay | una sola para todo el juego | un contador que baja con cada golpe |
| Cuando la nave es golpeada | se vuelve a centrar con `center_ship()` | se resta 1 |

### Try it Yourself

> 13-6. Game Over:
In Sideways Shooter, keep track of the number of times the ship is hit and the number of times an alien is hit by the ship. Decide on an appropriate condition for ending the game, and stop the game when this situation occurs.

In [ ]:
# 13-6


## Final Code

Los archivos del proyecto al terminar el capítulo (se ejecuta solo `alien_invasion.py`; las imágenes `ship.bmp` y `alien.bmp` van en la carpeta `images`).

**alien_invasion.py**

In [ ]:
# alien_invasion.py
import sys
from time import sleep

import pygame

from settings import Settings
from game_stats import GameStats
from ship import Ship
from bullet import Bullet
from alien import Alien


class AlienInvasion:
    """Overall class to manage game assets and behavior."""

    def __init__(self):
        """Initialize the game, and create game resources."""
        pygame.init()
        self.clock = pygame.time.Clock()
        self.settings = Settings()

        self.screen = pygame.display.set_mode(
            (self.settings.screen_width, self.settings.screen_height))
        pygame.display.set_caption("Alien Invasion")

        # Create an instance to store game statistics.
        self.stats = GameStats(self)

        self.ship = Ship(self)
        self.bullets = pygame.sprite.Group()
        self.aliens = pygame.sprite.Group()

        self._create_fleet()

        # Start Alien Invasion in an active state.
        self.game_active = True

    def run_game(self):
        """Start the main loop for the game."""
        while True:
            self._check_events()
            
            if self.game_active:
                self.ship.update()
                self._update_bullets()
                self._update_aliens()

            self._update_screen()
            self.clock.tick(60)

    def _check_events(self):
        """Respond to keypresses and mouse events."""
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                sys.exit()
            elif event.type == pygame.KEYDOWN:
                self._check_keydown_events(event)
            elif event.type == pygame.KEYUP:
                self._check_keyup_events(event)

    def _check_keydown_events(self, event):
        """Respond to keypresses."""
        if event.key == pygame.K_RIGHT:
            self.ship.moving_right = True
        elif event.key == pygame.K_LEFT:
            self.ship.moving_left = True
        elif event.key == pygame.K_q:
            sys.exit()
        elif event.key == pygame.K_SPACE:
            self._fire_bullet()            

    def _check_keyup_events(self, event):
        """Respond to key releases."""
        if event.key == pygame.K_RIGHT:
            self.ship.moving_right = False
        elif event.key == pygame.K_LEFT:
            self.ship.moving_left = False

    def _fire_bullet(self):
        """Create a new bullet and add it to the bullets group."""
        if len(self.bullets) < self.settings.bullets_allowed:
            new_bullet = Bullet(self)
            self.bullets.add(new_bullet)

    def _update_bullets(self):
        """Update position of bullets and get rid of old bullets."""
        # Update bullet positions.
        self.bullets.update()

        # Get rid of bullets that have disappeared.
        for bullet in self.bullets.copy():
            if bullet.rect.bottom <= 0:
                self.bullets.remove(bullet)

        self._check_bullet_alien_collisions()
        

    def _check_bullet_alien_collisions(self):
        """Respond to bullet-alien collisions."""
        # Remove any bullets and aliens that have collided.
        collisions = pygame.sprite.groupcollide(
                self.bullets, self.aliens, True, True)

        if not self.aliens:
            # Destroy existing bullets and create new fleet.
            self.bullets.empty()
            self._create_fleet()

    def _ship_hit(self):
        """Respond to the ship being hit by an alien."""
        if self.stats.ships_left > 0:
            # Decrement ships_left.
            self.stats.ships_left -= 1

            # Get rid of any remaining bullets and aliens.
            self.bullets.empty()
            self.aliens.empty()

            # Create a new fleet and center the ship.
            self._create_fleet()
            self.ship.center_ship()

            # Pause.
            sleep(0.5)
        else:
            self.game_active = False

    def _update_aliens(self):
        """Update the positions of all aliens in the fleet."""
        """Check if the fleet is at an edge, then update positions."""
        self._check_fleet_edges()
        self.aliens.update()

        # Look for alien-ship collisions.
        if pygame.sprite.spritecollideany(self.ship, self.aliens):
            self._ship_hit()

        # Look for aliens hitting the bottom of the screen.
        self._check_aliens_bottom()

    def _check_aliens_bottom(self):
        """Check if any aliens have reached the bottom of the screen."""
        for alien in self.aliens.sprites():
            if alien.rect.bottom >= self.settings.screen_height:
                # Treat this the same as if the ship got hit.
                self._ship_hit()
                break

    def _create_fleet(self):
        """Create the fleet of aliens."""
        # Create an alien and keep adding aliens until there's no room left.
        # Spacing between aliens is one alien width and one alien height.
        alien = Alien(self)
        alien_width, alien_height = alien.rect.size

        current_x, current_y = alien_width, alien_height
        while current_y < (self.settings.screen_height - 3 * alien_height):
            while current_x < (self.settings.screen_width - 2 * alien_width):
                self._create_alien(current_x, current_y)
                current_x += 2 * alien_width

            # Finished a row; reset x value, and increment y value.
            current_x = alien_width
            current_y += 2 * alien_height

    def _create_alien(self, x_position, y_position):
        """Create an alien and place it in the fleet."""
        new_alien = Alien(self)
        new_alien.x = x_position
        new_alien.rect.x = x_position
        new_alien.rect.y = y_position
        self.aliens.add(new_alien)

    def _check_fleet_edges(self):
        """Respond appropriately if any aliens have reached an edge."""
        for alien in self.aliens.sprites():
            if alien.check_edges():
                self._change_fleet_direction()
                break

    def _change_fleet_direction(self):
        """Drop the entire fleet and change the fleet's direction."""
        for alien in self.aliens.sprites():
            alien.rect.y += self.settings.fleet_drop_speed
        self.settings.fleet_direction *= -1

    def _update_screen(self):
        """Update images on the screen, and flip to the new screen."""
        self.screen.fill(self.settings.bg_color)
        for bullet in self.bullets.sprites():
            bullet.draw_bullet()
        self.ship.blitme()
        self.aliens.draw(self.screen)

        pygame.display.flip()


if __name__ == '__main__':
    # Make a game instance, and run the game.
    ai = AlienInvasion()
    ai.run_game()

**alien.py**

In [ ]:
# alien.py
import pygame

from pygame.sprite import Sprite


class Alien(Sprite):
    """A class to represent a single alien in the fleet."""

    def __init__(self, ai_game):
        """Initialize the alien and set its starting position."""
        super().__init__()
        self.screen = ai_game.screen
        self.settings = ai_game.settings

        # Load the alien image and set its rect attribute.
        self.image = pygame.image.load('images/alien.bmp')
        self.rect = self.image.get_rect()

        # Start each new alien near the top left of the screen.
        self.rect.x = self.rect.width
        self.rect.y = self.rect.height

        # Store the alien's exact horizontal position.
        self.x = float(self.rect.x)

    def check_edges(self):
        """Return True if alien is at edge of screen."""
        screen_rect = self.screen.get_rect()
        return (self.rect.right >= screen_rect.right) or (self.rect.left <= 0)

    def update(self):
        """Move the alien right or left."""
        self.x += self.settings.alien_speed * self.settings.fleet_direction
        self.rect.x = self.x

**bullet.py**

In [ ]:
# bullet.py
import pygame
from pygame.sprite import Sprite

class Bullet(Sprite):
    """A class to manage bullets fired from the ship."""

    def __init__(self, ai_game):
        """Create a bullet object at the ship's current position."""
        super().__init__()
        self.screen = ai_game.screen
        self.settings = ai_game.settings
        self.color = self.settings.bullet_color

        # Create a bullet rect at (0, 0) and then set correct position.
        self.rect = pygame.Rect(0, 0, self.settings.bullet_width,
            self.settings.bullet_height)
        self.rect.midtop = ai_game.ship.rect.midtop

        # Store the bullet's position as a float.
        self.y = float(self.rect.y)

    def update(self):
        """Move the bullet up the screen."""
        # Update the exact position of the bullet.
        self.y -= self.settings.bullet_speed
        # Update the rect position.
        self.rect.y = self.y

    def draw_bullet(self):
        """Draw the bullet to the screen."""
        pygame.draw.rect(self.screen, self.color, self.rect)

**game_stats.py**

In [ ]:
# game_stats.py
class GameStats:
    """Track statistics for Alien Invasion."""

    def __init__(self, ai_game):
        """Initialize statistics."""
        self.settings = ai_game.settings
        self.reset_stats()

    def reset_stats(self):
        """Initialize statistics that can change during the game."""
        self.ships_left = self.settings.ship_limit

**settings.py**

In [ ]:
# settings.py
class Settings:
    """A class to store all settings for Alien Invasion."""

    def __init__(self):
        """Initialize the game's settings."""
        # Screen settings
        self.screen_width = 1200
        self.screen_height = 800
        self.bg_color = (230, 230, 230)

        # Ship settings.
        self.ship_speed = 1.5
        self.ship_limit = 3

        # Bullet settings
        self.bullet_speed = 2.5
        self.bullet_width = 3
        self.bullet_height = 15
        self.bullet_color = (60, 60, 60)
        self.bullets_allowed = 3

        # Alien settings.
        self.alien_speed = 1.0
        self.fleet_drop_speed = 10
        # fleet_direction of 1 represents right; -1 represents left.
        self.fleet_direction = 1

**ship.py**

In [ ]:
# ship.py
import pygame


class Ship:
    """A class to manage the ship."""

    def __init__(self, ai_game):
        """Initialize the ship and set its starting position."""
        self.screen = ai_game.screen
        self.settings = ai_game.settings
        self.screen_rect = ai_game.screen.get_rect()

        # Load the ship image and get its rect.
        self.image = pygame.image.load('images/ship.bmp')
        self.rect = self.image.get_rect()

        # Start each new ship at the bottom center of the screen.
        self.rect.midbottom = self.screen_rect.midbottom

        # Store a float for the ship's exact horizontal position.
        self.x = float(self.rect.x)

        # Movement flags; start with a ship that's not moving.
        self.moving_right = False
        self.moving_left = False

    def center_ship(self):
        """Center the ship on the screen."""
        self.rect.midbottom = self.screen_rect.midbottom
        self.x = float(self.rect.x)

    def update(self):
        """Update the ship's position based on movement flags."""
        # Update the ship's x value, not the rect.
        if self.moving_right and self.rect.right < self.screen_rect.right:
            self.x += self.settings.ship_speed
        if self.moving_left and self.rect.left > 0:
            self.x -= self.settings.ship_speed
            
        # Update rect object from self.x.
        self.rect.x = self.x

    def blitme(self):
        """Draw the ship at its current location."""
        self.screen.blit(self.image, self.rect)

## Comparison: Chapter 13

| Si necesito... | Uso |
|---|---|
| Muchos elementos iguales | clase que hereda de `Sprite` + `pygame.sprite.Group()` |
| Dibujar todo un grupo | `grupo.draw(superficie)` |
| Una cuadrícula de elementos | dos `while` anidados (x dentro, y fuera) |
| Mover un grupo entero | `grupo.update()` y un `update()` en cada sprite |
| Cambiar de dirección | una variable que vale `1` o `-1`, multiplicada por la velocidad |
| Saber si un elemento llegó a un borde | comparar `rect.right`, `rect.left` y `rect.bottom` con los de la pantalla |
| Destruir elementos que chocan | `pygame.sprite.groupcollide(g1, g2, True, True)` |
| Saber si algo chocó con la nave | `pygame.sprite.spritecollideany(sprite, grupo)` |
| Saber si un grupo se quedó vacío | `if not grupo:` |
| Reiniciar los datos de la partida | `GameStats.reset_stats()` |
| Pausar un momento | `sleep(0.5)` |
| Terminar o pausar el juego | flag `game_active` |
| Probar más fácilmente | cambiar settings temporalmente (balas anchas, `fleet_drop_speed` alto) y restaurarlos |

| | `alien_invasion.py` | `alien.py` | `game_stats.py` |
|---|---|---|---|
| Clase | `AlienInvasion` | `Alien` | `GameStats` |
| Responsabilidad | main loop, flota, colisiones y fin del juego | un alien: imagen, movimiento y chequeo de bordes | estadísticas de la partida |

| Método de `AlienInvasion` | Qué hace |
|---|---|
| `_create_fleet()` | crea la flota (loops anidados) |
| `_create_alien()` | crea un alien en una posición |
| `_check_fleet_edges()` | revisa si algún alien llegó a un borde |
| `_change_fleet_direction()` | baja la flota y cambia la dirección |
| `_update_aliens()` | revisa bordes, mueve los aliens y detecta colisiones con la nave y el borde inferior |
| `_check_bullet_alien_collisions()` | destruye aliens golpeados y repone la flota |
| `_ship_hit()` | responde a que se perdió una nave |
| `_check_aliens_bottom()` | detecta aliens que llegaron abajo |